# Food Delivery Data Analysis – Internship Entrance Test

This notebook is created as part of the **Innomatics Research Lab Internship Entrance Test**.

The objective of this task is to simulate a real-world data engineering and analytics workflow by:
- Loading datasets from multiple file formats (CSV, JSON, SQL)
- Merging them into a single analytical dataset
- Performing data validation
- Answering analytical questions based solely on the final merged dataset

The final dataset generated in this notebook serves as the **single source of truth** for all subsequent analysis and questions.


## Mounting Google Drive

In this step, we mount Google Drive to the Colab environment.  
This allows us to access the dataset files (`orders.csv`, `users.json`, and `restaurants.sql`) that are stored in Google Drive and use them directly within the notebook.

Mounting the drive ensures a persistent and organized way to load external datasets required for data ingestion and analysis.


In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Data Loading and Initial Validation

In this section, we load the datasets provided for the analysis, each coming from a different file format to simulate real-world data sources:

- **orders.csv** – Transactional order-level data
- **users.json** – User master data containing demographic and membership information
- **restaurants.sql** – Restaurant master data containing cuisine type and ratings

The SQL file is executed using an in-memory SQLite database to accurately recreate the restaurant table before extracting it into a Pandas DataFrame.

After loading each dataset, basic shape checks and sample previews are performed to ensure the data has been loaded correctly and is ready for further processing.


In [2]:
# Import required libraries
import pandas as pd
import sqlite3
import json

In [3]:
# File paths
orders_path = "/content/drive/MyDrive/Datasets/IRL_DATASETS/orders.csv"
users_path = "/content/drive/MyDrive/Datasets/IRL_DATASETS/users.json"
restaurants_sql_path = "/content/drive/MyDrive/Datasets/IRL_DATASETS/restaurants.sql"

# 1️⃣ Load orders.csv
orders_df = pd.read_csv(orders_path)

# 2️⃣ Load users.json
with open(users_path, "r") as f:
    users_data = json.load(f)
users_df = pd.DataFrame(users_data)

# 3️⃣ Load restaurants.sql into SQLite and read as DataFrame
conn = sqlite3.connect(":memory:")  # in-memory DB

with open(restaurants_sql_path, "r") as f:
    sql_script = f.read()

conn.executescript(sql_script)

restaurants_df = pd.read_sql_query("SELECT * FROM restaurants", conn)

# Quick sanity check
print("Orders:", orders_df.shape)
print("Users:", users_df.shape)
print("Restaurants:", restaurants_df.shape)

orders_df.head(), users_df.head(), restaurants_df.head()


Orders: (10000, 6)
Users: (3000, 4)
Restaurants: (500, 4)


(   order_id  user_id  restaurant_id  order_date  total_amount  \
 0         1     2508            450  18-02-2023        842.97   
 1         2     2693            309  18-01-2023        546.68   
 2         3     2084            107  15-07-2023        163.93   
 3         4      319            224  04-10-2023       1155.97   
 4         5     1064            293  25-12-2023       1321.91   
 
                   restaurant_name  
 0               New Foods Chinese  
 1  Ruchi Curry House Multicuisine  
 2           Spice Kitchen Punjabi  
 3          Darbar Kitchen Non-Veg  
 4       Royal Eatery South Indian  ,
    user_id    name       city membership
 0        1  User_1    Chennai    Regular
 1        2  User_2       Pune       Gold
 2        3  User_3  Bangalore       Gold
 3        4  User_4  Bangalore    Regular
 4        5  User_5       Pune       Gold,
    restaurant_id restaurant_name  cuisine  rating
 0              1    Restaurant_1  Chinese     4.8
 1              2    Res

## Data Merging (Step 4)

In this step, the three datasets are merged to create a unified analytical dataset.

The joins are performed as follows:
- `orders.user_id` → `users.user_id`
- `orders.restaurant_id` → `restaurants.restaurant_id`

**Left joins** are used to ensure that all order records are retained, even if corresponding user or restaurant information is missing.

This approach preserves the integrity of transactional data and aligns with real-world analytical practices.


In [4]:
# Step 4.1: Merge orders with users (LEFT JOIN)
orders_users_df = pd.merge(
    orders_df,
    users_df,
    on="user_id",
    how="left"
)

# Step 4.2: Merge the above with restaurants (LEFT JOIN)
final_df = pd.merge(
    orders_users_df,
    restaurants_df,
    on="restaurant_id",
    how="left"
)

# Sanity checks
print("Final dataset shape:", final_df.shape)
final_df.head()


Final dataset shape: (10000, 12)


,order_id,user_id,restaurant_id,order_date,total_amount,restaurant_name_x,name,city,membership,restaurant_name_y,cuisine,rating
0,1,2508,450,18-02-2023,842.97,New Foods Chinese,User_2508,Hyderabad,Regular,Restaurant_450,Mexican,3.2
1,2,2693,309,18-01-2023,546.68,Ruchi Curry House Multicuisine,User_2693,Pune,Regular,Restaurant_309,Indian,4.5
2,3,2084,107,15-07-2023,163.93,Spice Kitchen Punjabi,User_2084,Chennai,Gold,Restaurant_107,Mexican,4.0
3,4,319,224,04-10-2023,1155.97,Darbar Kitchen Non-Veg,User_319,Bangalore,Gold,Restaurant_224,Chinese,4.8
4,5,1064,293,25-12-2023,1321.91,Royal Eatery South Indian,User_1064,Pune,Regular,Restaurant_293,Italian,3.0


## Column Cleanup and Date Formatting

After merging the datasets:
- Duplicate restaurant name columns created during the merge are resolved
- The order date column is converted to a proper datetime format

These steps ensure consistency, readability, and readiness for time-based analysis such as quarterly trends and seasonality.


In [5]:
final_df.isnull().sum()


,0
order_id,0
user_id,0
restaurant_id,0
order_date,0
total_amount,0
restaurant_name_x,0
name,0
city,0
membership,0
restaurant_name_y,0


In [6]:
final_df = final_df.drop(columns=["restaurant_name_y"])
final_df = final_df.rename(columns={"restaurant_name_x": "restaurant_name"})


In [7]:
final_df["order_date"] = pd.to_datetime(
    final_df["order_date"],
    format="%d-%m-%Y"
)


In [8]:
final_df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   order_id         10000 non-null  int64         
 1   user_id          10000 non-null  int64         
 2   restaurant_id    10000 non-null  int64         
 3   order_date       10000 non-null  datetime64[ns]
 4   total_amount     10000 non-null  float64       
 5   restaurant_name  10000 non-null  object        
 6   name             10000 non-null  object        
 7   city             10000 non-null  object        
 8   membership       10000 non-null  object        
 9   cuisine          10000 non-null  object        
 10  rating           10000 non-null  float64       
dtypes: datetime64[ns](1), float64(2), int64(3), object(5)
memory usage: 859.5+ KB


In [9]:
final_df["membership"].value_counts()
final_df["city"].value_counts().head()
final_df["cuisine"].value_counts()


,count
cuisine,
Mexican,2581
Italian,2532
Indian,2469
Chinese,2418


In [10]:
final_df["total_amount"].sum()


np.float64(8011624.12)

## Final Dataset Creation and Export

The cleaned and merged dataset is now finalized.

This dataset contains:
- Order details
- User information
- Restaurant information

The final dataset is exported as **`final_food_delivery_dataset.csv`** and serves as the **only source of truth** for all subsequent analysis and questions in this assessment.


In [11]:
# Step 5: Export final dataset
output_path = "/content/drive/MyDrive/Datasets/IRL_DATASETS/final_food_delivery_dataset.csv"
final_df.to_csv(output_path, index=False)

print("Final dataset saved at:", output_path)

Final dataset saved at: /content/drive/MyDrive/Datasets/IRL_DATASETS/final_food_delivery_dataset.csv


🟢 MCQ 1
Which city has the highest total revenue from Gold members?
What to compute

Filter → membership == "Gold"

Group by → city

Metric → sum(total_amount)

Pick max

In [12]:
(
    final_df[final_df["membership"] == "Gold"]
    .groupby("city")["total_amount"]
    .sum()
    .sort_values(ascending=False)
)


,total_amount
city,
Chennai,1080909.79
Pune,1003012.32
Bangalore,994702.59
Hyderabad,896740.19


🟢 MCQ 2
Which cuisine has the highest average order value across all orders?
What to compute

Group by → cuisine

Metric → mean(total_amount)

Pick max

In [13]:
(
    final_df
    .groupby("cuisine")["total_amount"]
    .mean()
    .sort_values(ascending=False)
)


,total_amount
cuisine,
Mexican,808.021344
Italian,799.448578
Indian,798.466011
Chinese,798.389020


How many distinct users placed orders worth more than ₹1000 in total?

⚠️ Careful: total per user, not per order.

What to compute

Group by user_id

Sum total_amount

Count users where sum > 1000

In [14]:
user_spend = (
    final_df
    .groupby("user_id")["total_amount"]
    .sum()
)

count_users = (user_spend > 1000).sum()
count_users


np.int64(2544)

🟢 MCQ 4
Which restaurant rating range generated the highest total revenue?

In [15]:
#Step 1: Create rating bins
rating_bins = pd.cut(
    final_df["rating"],
    bins=[3.0, 3.5, 4.0, 4.5, 5.0],
    right=True
)


In [32]:
#Step 2: Revenue by rating range
(
    final_df
    .assign(rating_range=rating_bins)
    .groupby("rating_range")["total_amount"]
    .sum()
)


/tmp/ipython-input-1452260212.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby("rating_range")["total_amount"]


,total_amount
rating_range,
"(3.0, 3.5]",1881754.57
"(3.5, 4.0]",1717494.41
"(4.0, 4.5]",1960326.26
"(4.5, 5.0]",2197030.75


🟢 MCQ 5
Among Gold members, which city has the highest average order value?
What to compute

Filter Gold

Group by city

Mean of total_amount

In [17]:
(
    final_df[final_df["membership"] == "Gold"]
    .groupby("city")["total_amount"]
    .mean()
    .sort_values(ascending=False)
)


,total_amount
city,
Chennai,808.459080
Hyderabad,806.421034
Bangalore,793.223756
Pune,781.162243


🟢 MCQ 6
Which cuisine has the lowest number of distinct restaurants but still contributes significant revenue?

This is a two-metric question.

In [18]:
#Step 1: Distinct restaurants per cuisine
restaurants_per_cuisine = (
    final_df
    .groupby("cuisine")["restaurant_id"]
    .nunique()
    .sort_values()
)

restaurants_per_cuisine


,restaurant_id
cuisine,
Chinese,120
Indian,126
Italian,126
Mexican,128


In [19]:
#Step 2: Revenue per cuisine
revenue_per_cuisine = (
    final_df
    .groupby("cuisine")["total_amount"]
    .sum()
    .sort_values(ascending=False)
)

revenue_per_cuisine


,total_amount
cuisine,
Mexican,2085503.09
Italian,2024203.80
Indian,1971412.58
Chinese,1930504.65


#🟢 MCQ 7

What percentage of total orders were placed by Gold members?

##Logic:

Total orders = len(final_df)

Gold orders = count where membership == "Gold"

Percentage = (Gold / Total) * 100

Round to nearest integer

In [20]:
gold_order_percentage = (
    (final_df["membership"] == "Gold").mean() * 100
)

round(gold_order_percentage)


50

#🟢MCQ 7
Which restaurant has the highest average order value but less than 20 total orders?



##🟢 Step 1: Compute restaurant-level stats

In [21]:
restaurant_stats = (
    final_df
    .groupby("restaurant_name")
    .agg(
        order_count=("order_id", "count"),
        avg_order_value=("total_amount", "mean")
    )
)


##🟢 Step 2: Apply the constraint + rank

In [22]:
filtered = (
    restaurant_stats[restaurant_stats["order_count"] < 20]
    .sort_values("avg_order_value", ascending=False)
)

filtered.head(10)


,order_count,avg_order_value
restaurant_name,,
Hotel Dhaba Multicuisine,13,1040.222308
Sri Mess Punjabi,12,1029.180833
Ruchi Biryani Punjabi,16,1002.140625
Sri Delights Pure Veg,18,989.467222
Classic Kitchen Family Restaurant,19,973.167895
Hotel Dhaba Chinese,18,973.125556
Amma Mess Pure Veg,18,965.299444
Hotel Biryani Pure Veg,13,964.577692
Annapurna Curry House Multicuisine,17,954.512353


##🟢 What to do now (final, correct step)

We must restrict to the four given restaurants and then re-check.

In [23]:
options = [
    "Grand Cafe Punjabi",
    "Grand Restaurant South Indian",
    "Ruchi Mess Multicuisine",
    "Ruchi Foods Chinese"
]

restaurant_stats.loc[options]


,order_count,avg_order_value
restaurant_name,,
Grand Cafe Punjabi,32,765.409063
Grand Restaurant South Indian,29,842.567586
Ruchi Mess Multicuisine,40,851.226250
Ruchi Foods Chinese,19,686.603158


##🟢 MCQ 9

Which combination contributes the highest revenue?

Options:

Gold + Indian cuisine

Gold + Italian cuisine

Regular + Indian cuisine

Regular + Chinese cuisine

In [24]:
(
    final_df
    .groupby(["membership", "cuisine"])["total_amount"]
    .sum()
    .sort_values(ascending=False)
)


membership  cuisine
Regular     Mexican    1072943.30
            Italian    1018424.75
Gold        Mexican    1012559.79
            Italian    1005779.05
Regular     Indian      992100.27
Gold        Indian      979312.31
            Chinese     977713.74
Regular     Chinese     952790.91
Name: total_amount, dtype: float64

##🟢 MCQ 10

During which quarter of the year is the total revenue highest?

In [25]:
final_df["quarter"] = final_df["order_date"].dt.to_period("Q")

(
    final_df
    .groupby("quarter")["total_amount"]
    .sum()
    .sort_values(ascending=False)
)


,total_amount
quarter,
2023Q3,2037385.10
2023Q4,2018263.66
2023Q1,1993425.14
2023Q2,1945348.72
2024Q1,17201.50


##Q1.How many total orders were placed by users with Gold membership?

In [26]:
(final_df["membership"] == "Gold").sum()


np.int64(4987)

##Q2.What is the total revenue (rounded to nearest integer) generated from orders placed in Hyderabad city?

In [27]:
hyderabad_revenue = (
    final_df[final_df["city"] == "Hyderabad"]["total_amount"]
    .sum()
)

round(hyderabad_revenue)


1889367

##Q3.How many distinct users placed at least one order?

In [28]:
final_df["user_id"].nunique()


2883

##Q4.What is the average order value (rounded to 2 decimals) for Gold members?

In [29]:
gold_aov = (
    final_df[final_df["membership"] == "Gold"]["total_amount"]
    .mean()
)

round(gold_aov, 2)


np.float64(797.15)

##Q5.How many orders were placed for restaurants with rating ≥ 4.5?

In [30]:
(final_df["rating"] >= 4.5).sum()


np.int64(3374)

##Q6.How many orders were placed in the top revenue city among Gold members only?

In [31]:
(
    (final_df["membership"] == "Gold") &
    (final_df["city"] == "Chennai")
).sum()


np.int64(1337)